In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, precision_score

#STEP 1: DATA INGESTION & CLEANING
# Using 'latin-1' to fix the UnicodeDecodeError from previous attempts
try:
    df = pd.read_csv('spam.csv', encoding='latin-1')
    df = df.dropna(how="any", axis=1) # Remove empty columns
    df.columns = ['label', 'content']
    print("Success: Dataset loaded and cleaned.")
except FileNotFoundError:
    print("Error: 'spam.csv' not found. Please ensure the file is in the same directory.")

#STEP 2: MULTI-DIMENSIONAL FEATURE EXTRACTION
# Dimension A: Structural Features (Length, Links, Keywords)
df['email_len'] = df['content'].apply(len)
df['link_count'] = df['content'].apply(lambda x: len(re.findall(r'http|www', str(x))))
df['has_attachment'] = df['content'].apply(lambda x: 1 if 'attachment' in str(x).lower() else 0)

# Dimension B: Text Features (TF-IDF Word Frequency)
tfidf = TfidfVectorizer(stop_words='english', max_features=1000)
X_text = tfidf.fit_transform(df['content']).toarray()

#STEP 3: PRE-PROCESSING (SCALING & ENCODING)
# Normalize numerical features so they have a mean of 0 and variance of 1
scaler = StandardScaler()
X_num = scaler.fit_transform(df[['email_len', 'link_count', 'has_attachment']])

# Combine all dimensions into one feature matrix
X = np.hstack((X_text, X_num))

# Encode the output variable (Ham=0, Spam=1)
le = LabelEncoder()
y = le.fit_transform(df['label'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# STEP 4: COMPARATIVE MODELING
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM (Linear Kernel)": SVC(kernel='linear', probability=True)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Report": classification_report(y_test, y_pred, target_names=le.classes_)
    }

#STEP 5: COMPARATIVE ANALYSIS & DECISION SUPPORT
print("\n" + "="*30)
print("COMPARATIVE ANALYSIS REPORT")
print("="*30)

for name, metrics in results.items():
    print(f"\nMODEL: {name}")
    print(f"Overall Accuracy: {metrics['Accuracy']:.2%}")
    print(f"Spam Precision: {metrics['Precision']:.2%}")
    print(metrics['Report'])


best_model = max(results, key=lambda x: results[x]['Precision'])
print(f"\nRECOMMENDATION: Deploy {best_model}.")
print(f"Reasoning: {best_model} provides the highest Precision, which minimizes the risk of marking legitimate business emails as spam.")


Success: Dataset loaded and cleaned.

COMPARATIVE ANALYSIS REPORT

MODEL: Logistic Regression
Overall Accuracy: 96.41%
Spam Precision: 93.65%
              precision    recall  f1-score   support

         ham       0.97      0.99      0.98       965
        spam       0.94      0.79      0.86       150

    accuracy                           0.96      1115
   macro avg       0.95      0.89      0.92      1115
weighted avg       0.96      0.96      0.96      1115


MODEL: SVM (Linear Kernel)
Overall Accuracy: 97.94%
Spam Precision: 97.04%
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       965
        spam       0.97      0.87      0.92       150

    accuracy                           0.98      1115
   macro avg       0.98      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115


RECOMMENDATION: Deploy SVM (Linear Kernel).
Reasoning: SVM (Linear Kernel) provides the highest Precision, which minimizes the 